> **Series**: *Bayesian Modelling for Industrial Applications*, Part 3. [Part 1](https://sambaiga.github.io/pages/blog/posts/2025/10/2025-10-10-bayesion-foundation.html) introduced Bayesian thinking, and [Part 2](https://sambaiga.github.io/pages/blog/posts/2025/12/2025-12-1-bayesian-regression..html) fitted one Beta regression to the six cells of one chemistry.

## Limits of One Curve

Part 2 gave the six cells of one chemistry the same degradation curve. That is a sensible first model, and its test showed the limit: predicted from the other five, four cells were followed closely and two were missed. Cells differ, and cells of one chemistry resemble each other more than they resemble cells of another. A manufacturer that tests a few cells of a new chemistry has to decide how far to trust one common curve, how far to trust the record of each cell, and what to believe about a cell that has just been installed.

This post answers those questions with a hierarchical model, in which each cell draws its parameters from its chemistry and each chemistry from the whole fleet. The model is built with PyMC[^pymc] and fitted to the CALCE cycling data of Part 2, now with both chemistries. It is judged against its two obvious alternatives, one curve for all cells and one curve per cell, on two cells that the model never saw, and against a control that has no diagnostics at all. The hierarchy separates the two chemistries sharply. As a predictor for a new cell it helps in one setting only, after the cell has shown its first 100 cycles, and the post reports where it does not.

[^pymc]: O. Abril-Pla et al., PyMC: a modern, and comprehensive probabilistic programming framework in Python, *PeerJ Computer Science* 9, e1516, 2023.

## Levels of Pooling

Three strategies are compared throughout the post. Under **complete pooling** every cell shares one set of curve parameters, so a new cell receives the average curve of the fleet and the chemistry plays no part. Under **no pooling** each cell has parameters of its own and nothing is shared, so a cell is described by its own record only, and a cell without a record is described by the prior. **Partial pooling** lies between them: the parameters of a cell come from a distribution that belongs to its chemistry, and the parameters of a chemistry come from a distribution over the fleet. A cell with a long record is then free to move away from its chemistry, and a cell with a short or incomplete record is drawn towards it.

The choice matters most when records are short or incomplete, because the data then say little about a cell and the structure of the model decides what is believed. The sections that follow measure that on a real fleet, and the first of them describes the data.

## Thirteen Cells, Two Chemistries

The data are the cycle-by-cycle diagnostics of 13 lithium-ion cells from the CALCE laboratory, in the form prepared for Part 2.[^calce] Six cells are of chemistry CS2, with a nominal capacity of 1.10 Ah, and seven are of chemistry CX2, with 1.35 Ah. Every cell was cycled until it fell well below the end-of-life threshold of 80% state of health (SoH). The target is the SoH, the capacity divided by the nominal capacity, divided again by 1.1 so that it stays inside the open unit interval that a Beta likelihood requires. SoH exceeds 100% in 1.7% of the cycles, which the ceiling of 1.1 accommodates. The inputs are the four diagnostics of Part 2, standardised on the training cells only.

Every eighth cycle is kept, because neighbouring cycles are almost duplicates, which leaves 1,752 observations with between 78 and 196 per cell. Two cells, CS2_38 and CX2_38, the last of each chemistry, are held out to test the strategies on cells that the model has not seen, and the remaining 11 cells provide 1,456 observations for training. The inputs are measured in the same cycle as the capacity, so the model estimates the state of health from diagnostics and does not forecast it. A forecast would need those diagnostics to be predicted as well.

[^calce]: The feature file is downloaded from <https://ndownloader.figshare.com/files/59415941>. The underlying cycling experiments are those of the CALCE Battery Research Group, University of Maryland.

In [ ]:
# | output: false
from importlib.metadata import version
import os
from pathlib import Path
import tempfile
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "2")

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
from sklearn.preprocessing import StandardScaler

from bayes import priors
from bayes.data import calce
from bayes.plot import change, hierarchy, tables
from bayes.plot.inline import show
from bayes.regression import evaluate, hierarchical as hier

FAST = os.environ.get("HIER_FAST") == "1"  # a smoke test with a few draws, to check that the notebook runs
CACHE = Path("cache-fast" if FAST else "cache")
CACHE.mkdir(exist_ok=True)
FEATURES = ["charge_current_auc", "charge_current_mean", "discharge_voltage_auc", "discharge_voltage_crest"]
HELD_OUT = ["CS2_38", "CX2_38"]  # the last cell of each chemistry, kept back for the tests of new cells
SEEN = (0, 100, 300)  # cycles of a new cell that the model has seen
GAP = 50  # embargo, in cycles, between the cycles seen and the cycles scored
MAIN = {"draws": 40 if FAST else 500, "tune": 40 if FAST else 750, "chains": 2 if FAST else 4}
REFIT = {"draws": 40 if FAST else 500, "tune": 40 if FAST else 750, "chains": 2 if FAST else 4}
print("pymc", version("pymc"), "| arviz", version("arviz"), "| fast mode:", FAST)

In [ ]:
# | output: false
data = calce.load(Path(tempfile.gettempdir()) / "calce-battery-features.parquet", FEATURES, every=8)
train, held_out = calce.split(data, HELD_OUT)
scaler = StandardScaler().fit(train[FEATURES])  # standardised on the training cells only
CHEMISTRIES = list(dict.fromkeys(data["CellType"]))
COLOURS = hierarchy.palette(CHEMISTRIES)

summary = data.groupby(["CellType", "cell"]).agg(rows=("cycle", "size"), last_cycle=("cycle", "max"))
for chemistry, rows in summary.groupby("CellType"):
    print(
        f"{chemistry}: {len(rows)} cells, {rows['rows'].min()} to {rows['rows'].max()} kept cycles per cell, longest life {rows['last_cycle'].max()} cycles"
    )
print(
    f"{len(data):,} cycles kept; {len(train):,} in the {train['cell'].nunique()} training cells; held out: {', '.join(HELD_OUT)}"
)
print(f"SoH above 100%: {(data['soh'] > 100).mean():.1%} of the kept cycles")

In [ ]:
# | label: fig-curves
# | fig-cap: "State of health of the 13 cells against cycle. CX2 cells (orange) last longer and decline more slowly than CS2 cells (blue), dashed lines are the two cells held out for testing, and the dotted line is the end-of-life threshold of 80%."
show(hierarchy.curves(data, HELD_OUT, COLOURS))

The two chemistries differ markedly, as @fig-curves shows. The CX2 cells last up to 2,474 cycles and decline slowly, while the CS2 cells fail between 737 and 1,056 cycles. The records are not equally complete. CX2_16 begins at cycle 463 with a state of health of 94%, and CX2_33 begins at cycle 187, so for these two cells the early part of the curve is not observed. What the data say about their initial level is little, and a model that sees only these cells has to rely on its prior.

## The Hierarchical Model

The observation layer is a Beta regression, as in Part 2.[^beta] The state of health $y_{ij}$ of cell $j$ at its cycle $c_{ij}$ follows a Beta distribution with mean $\mu_{ij}$ and precision $\phi$,

$$
y_{ij} \sim \mathrm{Beta}\big(\mu_{ij}\phi,\ (1-\mu_{ij})\phi\big), \qquad \mu_{ij} = \mathrm{logit}^{-1}(\eta_{ij}),
$$

and the linear predictor on the logit scale combines a saturating exponential, which describes ageing, with the diagnostics $\mathbf{x}_{ij}$,

$$
\eta_{ij} = \alpha_j - A_k\left(1 - e^{-\lambda_j c_{ij}}\right) + \mathbf{x}_{ij}^{\top}\boldsymbol{\beta}.
$$

Here $\alpha_j$ is the initial level of cell $j$, $\lambda_j$ its rate, $A_k$ the amplitude of the chemistry $k$ to which it belongs, and the cycle is counted in thousands. The hierarchy ties the cell parameters to the chemistry. Each chemistry has an initial level $a_k$, a log rate $\ell_k$ and an amplitude $A_k$, drawn from distributions that are shared by the fleet, and a cell departs from its chemistry by an offset,

$$
\alpha_j = a_k + \sigma_{a,k}\, z_{a,j}, \qquad \log\lambda_j = \ell_k + \sigma_{\ell,k}\, z_{\ell,j}, \qquad z_{a,j},\, z_{\ell,j} \sim \mathcal{N}(0, 1).
$$

The unit-normal offsets $z$ are sampled and scaled afterwards, which is known as a non-centred parameterisation. It matters when the spread between cells is small, because a model that samples $\alpha_j$ directly then has a funnel-shaped posterior that the sampler traverses poorly and reports as divergences.[^funnel]

The code defines one function for the three strategies, so that they differ in nothing else. For complete pooling the three parameters are single numbers, for no pooling each cell has its own with an independent prior, and for partial pooling they follow the hierarchy above. The cell below builds the three models.

[^beta]: S. Ferrari and F. Cribari-Neto, Beta regression for modelling rates and proportions, *Journal of Applied Statistics* 31(7), 2004, pp. 799 to 815.

[^funnel]: M. Betancourt and M. Girolami, Hamiltonian Monte Carlo for hierarchical models, in *Current Trends in Bayesian Methodology with Applications*, CRC Press, 2015.

In [ ]:
# | output: false
models = {pooling: hier.hierarchical_model(train, scaler, FEATURES, pooling) for pooling in hier.POOLING}
print({pooling: len(model.free_RVs) for pooling, model in models.items()})

## Priors and Prior Checks

The priors are weakly informative, which means that they rule out values that no battery could show and leave the rest to the data.[^gelmanprior] The initial level has $\mathcal{N}(2, 1)$ at the fleet level, which puts the mean of the state of health near 97% when the other terms are zero, and the amplitudes have a half-normal prior. The log rate is centred on $\log 5$ per thousand cycles, with a standard deviation of 0.5. The standard deviations between cells and between chemistries have half-normal priors, as is usual for scale parameters. The effect of each standardised diagnostic has $\mathcal{N}(0, 0.2^2)$, so that a feature has to earn a large coefficient. The precision $\phi$ has a log-normal prior centred on 100, which corresponds to a noise standard deviation of about 5 points of state of health at mid-life. @tbl-priors lists them.

A prior predictive check shows what these choices imply before any data are seen. @fig-prior compares the state of health that the prior produces with the observed values of the training cells. The prior has its 5th, 50th and 95th percentiles at 23.6%, 91.1% and 109.1%, against 50.8%, 81.9% and 97.7% in the data. It is wider than the data at both ends and its median lies nine points above the observed one, and some of its mass exceeds 100%, which the data also do in 1.7% of the cycles. A weak prior should rule out the impossible and leave the rest to the data, and this one does.

[^gelmanprior]: A. Gelman, Prior distributions for variance parameters in hierarchical models, *Bayesian Analysis* 1(3), 2006, pp. 515 to 534.

In [ ]:
# | label: tbl-priors
# | tbl-cap: "Priors of the hierarchical Beta regression. The fleet level and the spreads are shared, and each cell draws its parameters from its chemistry through the cell spreads."
display(tables.themed_gt(priors.hierarchical()))

In [ ]:
# | label: fig-prior
# | fig-cap: "Observed state of health of the training cells and the prior predictive distribution of the model. The prior is wider than the data at both ends, with some mass above 100%."
with models["partial"]:
    prior = pm.sample_prior_predictive(draws=500, random_seed=1)
prior_soh = prior.prior_predictive["y"].to_numpy().ravel() * hier.CEILING * 100
drawn = np.random.default_rng(1).choice(prior_soh, size=min(20000, prior_soh.size), replace=False)
show(hierarchy.prior(train["soh"].to_numpy(), drawn))

## Posterior Sampling

The posteriors are sampled with the No-U-Turn sampler.[^nuts] Every fit runs four chains of 500 draws after 750 tuning draws, with a target acceptance rate of 0.90, and this covers the three models of the training cells and the eight refits of the test below. Sampling is the slow part. The eight refits took 34 minutes on a laptop, the hierarchical ones taking about 8 minutes each, so the notebook stores each fit in a `cache` folder and reads it back on a repeat run.

[^nuts]: M. D. Hoffman and A. Gelman, The No-U-Turn Sampler: adaptively setting path lengths in Hamiltonian Monte Carlo, *Journal of Machine Learning Research* 15, 2014, pp. 1593 to 1623.

In [ ]:
# | output: false
fits = {
    pooling: hier.sample(models[pooling], CACHE / f"train-{pooling}.nc", seed=2026 + index, log_likelihood=True, **MAIN)
    for index, pooling in enumerate(hier.POOLING)
}
for pooling, idata in fits.items():
    report = hier.convergence(idata)
    print(
        f"{pooling}: {idata.posterior.sizes['chain']} chains of {idata.posterior.sizes['draw']} draws, {report['divergences']} divergences, max R-hat {report['r_hat']:.3f}, min ESS bulk {report['ess_bulk']:.0f}, min ESS tail {report['ess_tail']:.0f}"
    )

### Convergence of the Chains

None of the 12 fits of the post, the three on the training cells, the eight refits and the control, has a divergent transition. For the three fits on the training cells the worst $\hat{R}$, the disagreement between chains, is 1.000 for partial pooling, 1.010 for complete pooling and 1.020 for no pooling, against a common guide of 1.01.[^rhat] The smallest bulk effective sample sizes are 820, 1,008 and 392, so the no-pooling model is the least well behaved. Its estimates are less precise than the others, and there is no sign of failure. Across all 12 fits the worst $\hat{R}$ is 1.02 and the smallest bulk effective sample size is 392, so the scores below do not rest on unconverged fits. The hierarchical model, which has the most parameters, samples well.

[^rhat]: A. Vehtari, A. Gelman, D. Simpson, B. Carpenter and P.-C. Bürkner, Rank-normalization, folding, and localization: an improved $\hat{R}$ for assessing convergence of MCMC, *Bayesian Analysis* 16(2), 2021, pp. 667 to 718.

## Posterior of the Chemistries

The chemistry level of the hierarchical model is where the model says something that a single curve cannot. @fig-chemistry shows the posterior of the three parameters for each chemistry. They differ in all three. The initial level is 1.19 for CS2 (90% interval 1.08 to 1.32) and 1.98 for CX2 (1.72 to 2.22) on the logit scale. The amplitude of the saturating term is 2.76 for CS2 (2.33 to 3.29) and 1.03 for CX2 (1.01 to 1.06). The log rate is $-0.31$ for CS2 (from $-0.78$ to 0.30) and 0.73 for CX2 (from 0.44 to 1.05), and the posterior probability that the rate of CX2 exceeds that of CS2 is 0.992.

In [ ]:
# | label: fig-chemistry
# | fig-cap: "Posterior mean and 90% interval of three parameters for each chemistry. CX2 starts higher, saturates faster and has a smaller amplitude than CS2."
show(hierarchy.chemistry(hier.chemistry_intervals(fits["partial"]), COLOURS))

Read together, the three describe the curves of @fig-curves. The CX2 term is small and saturates early, with a half-time of about 330 cycles, so the long, slow decline of those cells is carried by the diagnostics. The CS2 term is large and slow, with a half-time of about 950 cycles, so those cells keep losing capacity over their whole life. The difference between the chemistries is larger than the difference between cells within one. The standard deviation between cells of the log rate is 0.52 in CS2 and 0.47 in CX2, against a gap of 1.04 between the chemistries, and for the initial level it is 0.14 and 0.34, against a gap of 0.79. Only two chemistries inform these standard deviations, so they are set partly by their priors and are not interpreted further.

The remaining parameters are less eventful. Of the four diagnostics, only the area under the discharge voltage has an effect that differs from zero beyond doubt, at 0.362 with a posterior standard deviation of 0.016. The two charging-current features have effects of $-0.217$ and $-0.182$ with standard deviations of 0.142 each, and the crest factor of the discharge voltage has $-0.009 \pm 0.012$. The precision $\phi$ is about 1,800, which corresponds to a noise standard deviation of about one point of state of health. That is far in the upper tail of its prior, which puts 0.2% of its mass above 1,800, and it says that one diagnostic nearly determines the state of health. This has consequences for the tests that follow.

In [ ]:
# | label: fig-shrinkage
# | fig-cap: "Estimates of the initial level and of the log rate of each training cell without pooling (hollow) and with partial pooling (filled), with the posterior mean of each chemistry dashed. The initial level of CX2_16, whose record starts at cycle 463, moves most."
chemistry_of = train.drop_duplicates("cell").set_index("cell")["CellType"]
cells = [cell for chemistry in CHEMISTRIES for cell in chemistry_of[chemistry_of == chemistry].index]
estimates, means = hier.shrinkage_frames(fits["none"], fits["partial"], chemistry_of, cells)
show(hierarchy.shrinkage(estimates, means, COLOURS, cells))

### Cell Shrinkage

The hierarchy also changes what is said about a single cell. @fig-shrinkage sets the estimate of each training cell without pooling, hollow, beside the estimate with partial pooling, filled, with the posterior mean of the chemistry as a dashed line. The largest change is for the cell with the least complete record. The initial level of CX2_16, whose record begins at cycle 463, moves from 3.54 without pooling to 1.96 with it, close to the 1.98 of its chemistry, and CX2_33, which begins at cycle 187, moves from 2.09 to 1.74. The other cells change by 0.12 or less. Without pooling, the initial level of a cell with no early record is set by the prior and by observations far from cycle zero, and the hierarchy replaces that with what its chemistry shows.

The log rate changes more widely, by up to 0.94 in CS2_35 and 0.92 in CS2_33, although those cells have complete records. On a curve that has not yet saturated, the rate and the amplitude trade off against each other, so a cell's rate is poorly determined on its own, and the shared amplitude of its chemistry fixes part of that trade. This reading is consistent with the figure but was not tested separately. Averaged over the training cells, an estimate differs by 0.24 logit units for the initial level and by 0.40 for the log rate between the two strategies.

## Test on Unseen Cells

The test gives each strategy a new cell and asks for its state of health over the rest of its life. A new cell has no parameters in the fitted model, so its curve cannot be read from the posterior. The parameters are drawn instead from what each strategy says about a new cell. Under partial pooling the cell takes the chemistry level of its type and adds the offsets $\sigma_{a,k} z_a$ and $\sigma_{\ell,k} z_\ell$ with fresh unit-normal draws, which carries the cell-to-cell spread into the prediction. Under complete pooling it takes the common parameters, and under no pooling it takes its prior, since no pooled information exists. In each case the diagnostics of the cell go through the fitted effects, and the noise of the Beta likelihood is added to give predictive draws.

The strategies are first given no cycle of the new cell, and then its first 100 or 300 cycles, which are added to the training data before the models are refitted. All settings are scored on the same cycles, those after cycle 350 of each held-out cell, 222 observations in all, so that the scores differ only because of what was seen. Each setting is scored with the error of the median in points of state of health, the coverage of the 90% interval and the interval score, which is the width of the interval plus $2/\alpha$ times the distance by which an observation falls outside it, with $\alpha = 0.1$ and averaged over the observations. The interval score is a proper score, so it cannot be lowered by an interval that is too wide or too narrow for the truth.[^gneiting] A fourth model, complete pooling without the diagnostics, serves as a control: if the diagnostics carry the information, removing them should hurt.

[^gneiting]: T. Gneiting and A. E. Raftery, Strictly proper scoring rules, prediction, and estimation, *Journal of the American Statistical Association* 102(477), 2007, pp. 359 to 378.

In [ ]:
# | output: false
scores, diagnostics, curves = evaluate.cold_start(train, held_out, scaler, FEATURES, SEEN, GAP, MAIN, REFIT, CACHE)

### Prediction without Cycles

@fig-predict shows the three predictions for CS2_38 and CX2_38 when the model has seen none of their cycles, with the median as a line and the 90% interval as a band.

In [ ]:
# | label: fig-predict
# | fig-cap: "Predicted state of health of the two held-out cells before the model has seen any of their cycles. The line is the median and the band the 90% interval of each strategy, and the points are the observations."
panels = {"none": "(a) No pooling", "complete": "(b) Complete pooling", "partial": "(c) Partial pooling"}
shown = curves[curves["strategy"].isin(panels)].assign(method=lambda rows: rows["strategy"].map(panels))
show(hierarchy.predictions(shown.rename(columns={"soh": "observed"})))

Complete pooling predicts both cells well. Its median is off by 3.0 points of state of health on average, its interval is 17.1 points wide, and the interval contains 95.5% of the observations. Partial pooling is less accurate and wider, with a median error of 4.7 points, an interval 31.5 points wide and a coverage of 100%. No pooling returns its prior, with a median error of 6.6 points and an interval 73.6 points wide. The control without diagnostics has a median error of 6.8 points and an interval score of 60.7, against 18.6 for complete pooling, so the diagnostics cut the error by more than half and the interval score to about a third.

The ranking is the opposite of what the hierarchy was built for, and it has a reason. One diagnostic nearly determines the state of health, so the average curve with the right diagnostics is already close. A hierarchical prediction adds the spread between cells on top of that, which makes the interval wide. The coverage of 100% against a nominal 90% means that the interval is conservative, not that it is accurate.

In [ ]:
# | label: fig-change
# | fig-cap: "Interval score of each strategy as the change in percent from complete pooling, for each number of cycles seen. Orange bars are worse than the reference, and partial pooling is better only after 100 cycles."
ORDER = ["partial", "none", "control"]
shown = evaluate.change_by_strategy(scores, "interval", "complete", ORDER, hierarchy.STRATEGIES)
show(
    change.change(shown, "Complete pooling", [hierarchy.STRATEGIES[name] for name in ORDER], subject="Partial pooling")
)

In [ ]:
# | label: fig-coverage
# | fig-cap: "Coverage of the 90% intervals minus the nominal level, in percentage points, for each number of cycles seen. Orange bars fall short by more than 10 points. Partial and no pooling are over-confident after 300 cycles."
gap = evaluate.coverage_gap(
    scores, 1 - evaluate.ALPHA, ["partial", "complete", "none", "control"], hierarchy.STRATEGIES, subject="partial"
)
show(
    change.change(
        gap,
        "Nominal coverage, 90%",
        [hierarchy.STRATEGIES[name] for name in ("partial", "complete", "none", "control")],
        ncol=3,
        unit=" pts",
    )
)

@fig-change expresses the interval score of each strategy as the change from complete pooling, and @fig-coverage shows how far the coverage of each strategy is from the nominal 90%. Complete pooling is the reference because it has no cell-specific parameters to update: its error stays between 2.99 and 3.04 points, its coverage near 95% (95.5%, 95.0% and 94.1%) and its interval score at about 18.6 in every setting. The control without diagnostics is worse than the reference by more than 220% in every panel, with a median error of 6.8 to 6.9 points, which confirms that the diagnostics carry the information and that the other strategies are not being rewarded for something else.

The cell-specific strategies behave differently. After 100 cycles, no pooling has the smallest error, 1.6 points, but a coverage of 100% with an interval 22 points wide, which is mostly its prior, so its interval score is 18% above the reference. Partial pooling has an error of 3.2 points and a coverage of 94.6%, so it matches the calibration of complete pooling with an interval of 9.9 points against 17.0. Its interval score is 10.3 against 18.7, 45% below the reference, the one result in the test in which the hierarchy beats the single curve by a wide margin. With no cycle seen it is 69% above the reference, because its interval is too wide, 31.5 points with a coverage of 100%. After 300 cycles the picture reverses. Partial pooling and no pooling have intervals of about 5 points and cover only 52% and 29% of the observations, which is 38 and 61 points below the nominal level, so they are over-confident, and their interval scores are 133% and 109% above the reference. Both have learnt the early life of the cell and extrapolate it with a narrow band. The Beta likelihood treats the residuals of successive cycles as independent noise of about one point, whereas the real residuals of one cell are strongly correlated along its life. More data therefore made the calibration worse, which is a sign that the observation model is too simple and not that partial pooling failed.

Two held-out cells are a small test, and these differences should be read as observations and not as estimates of a general effect.

## Fit on Training Cells

Leave-one-out cross-validation, computed with Pareto-smoothed importance sampling,[^loo] answers a different question from the test above: how well each model predicts an observation that it has not been given, among the cells that it has seen. @fig-loo shows the loss in expected log predictive density against the best model. No pooling is best, partial pooling is 125 units behind it (standard error 14.5), and complete pooling is 2,242 units behind (standard error 76). The effective number of parameters is 63.6 for no pooling, 45.9 for partial pooling and 10.5 for complete pooling. One observation of the no-pooling model and one of the partial-pooling model have a Pareto $k$ above 0.7, and none of the complete-pooling model does.

[^loo]: A. Vehtari, A. Gelman and J. Gabry, Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC, *Statistics and Computing* 27, 2017, pp. 1413 to 1432.

In [ ]:
# | label: fig-loo
# | fig-cap: "Loss in expected log predictive density against the best model, from leave-one-out cross-validation within the training cells. Complete pooling is far behind, and partial pooling is 125 units behind no pooling."
labels = {"none": "No pooling", "complete": "Complete pooling", "partial": "Partial pooling"}
comparison = az.compare({labels[pooling]: fits[pooling] for pooling in hier.POOLING}, ic="loo").reset_index(
    names="method"
)
show(hierarchy.loo(comparison, best=comparison["method"].iloc[0]))

The ranking agrees with the previous section in one respect and disagrees in another. Cell-specific parameters matter a great deal for cells that are in the data, and the likelihood is so sharp that an offset of two or three points of state of health is worth many units of log density. The same parameters are not available for a new cell, which is why complete pooling is not last when the test is a new cell. Leave-one-out here leaves out single cycles, not whole cells, so it does not measure prediction for a new cell. The earlier tests do.

In [ ]:
# | output: false
# Numbers that the text quotes. This cell prints them and is hidden on the page.
posterior = fits["partial"].posterior
print(
    f"Prior predictive SoH percentiles 5/50/95: {np.percentile(prior_soh, [5, 50, 95]).round(1)}; observed {np.percentile(train['soh'], [5, 50, 95]).round(1)}"
)
for quantity, draws in (
    ("log rate", posterior["loglambda_chem"]),
    ("amplitude", posterior["amplitude_chem"]),
    ("alpha", posterior["alpha_chem"]),
):
    first, second = (draws.sel(chemistry=name).to_numpy().ravel() for name in CHEMISTRIES)
    print(
        f"{quantity}: {CHEMISTRIES[0]} {first.mean():.2f} ({np.percentile(first, 5):.2f} to {np.percentile(first, 95):.2f}), {CHEMISTRIES[1]} {second.mean():.2f} ({np.percentile(second, 5):.2f} to {np.percentile(second, 95):.2f}), P({CHEMISTRIES[1]} > {CHEMISTRIES[0]}) = {(second > first).mean():.3f}"
    )
print(
    "beta (mean, sd):",
    {
        name: (
            round(float(posterior["beta"].sel(feature=name).mean()), 3),
            round(float(posterior["beta"].sel(feature=name).std()), 3),
        )
        for name in FEATURES
    },
)
print(
    "phi mean",
    round(float(posterior["phi"].mean()), 1),
    "| sigma_cell_alpha",
    posterior["sigma_cell_alpha"].mean(("chain", "draw")).to_numpy().round(2),
    "| sigma_cell_lambda",
    posterior["sigma_cell_lambda"].mean(("chain", "draw")).to_numpy().round(2),
)
control = az.from_netcdf(CACHE / "train-control.nc")
print("control (complete pooling, no diagnostics): phi mean", round(float(control.posterior["phi"].mean()), 1))
print(
    "mean absolute distance between no pooling and partial pooling:",
    estimates.pivot_table(index=["parameter", "cell"], columns="method", values="mean")
    .assign(gap=lambda rows: (rows["No pooling"] - rows["Partial pooling"]).abs())
    .groupby("parameter")["gap"]
    .mean()
    .round(3)
    .to_dict(),
)
print(scores.round(3).to_string(index=False))
print(diagnostics.round(3).to_string(index=False))
print(comparison[["method", "elpd_loo", "elpd_diff", "dse", "p_loo"]].round(1).to_string(index=False))
print({name: int((az.loo(idata, pointwise=True).pareto_k > 0.7).sum()) for name, idata in fits.items()})

## Limits of the Study

- **The inputs are contemporaneous.** The diagnostics are measured in the same cycle as the capacity, so the study concerns estimation of the state of health, and not forecasting. The conclusions about pooling may differ for a forecast.
- **The test is small.** Two cells, one of each chemistry, are held out, and every coverage figure rests on two curves. A different pair, or a cross-validation over cells, would give a firmer answer.
- **The residuals are treated as independent.** The over-confidence after 300 cycles points to correlation of the residuals within a cell. A model with an autoregressive error or a cell-level smooth term would be the next step.
- **The precision prior is tight.** The posterior of $\phi$ lies far in the upper tail of its prior. With 1,456 observations the data dominate, but a wider prior should be checked.
- **Two chemistries are few.** The spread between chemistries is estimated from two groups, so the chemistry-level standard deviations are set partly by their priors and are not interpreted here.
- **The curve is simple.** A saturating exponential with a linear effect of the diagnostics has no knee, which the end of life of these cells shows.
- **Every eighth cycle is used.** This shortens the run and discards little, but the observations that remain are still correlated.

## Summary of Findings

A hierarchical model with cells in chemistries separates what belongs to a chemistry from what belongs to a cell. The two chemistries differ in all three curve parameters, with a probability of 0.992 that CX2 has the higher rate, and the spread between cells within a chemistry is smaller than the gap between chemistries. Partial pooling also steadies the cell whose early life is missing, CX2_16, whose initial level falls from 3.54 to 1.96.

As a predictor for new cells, the hierarchy is better than one common curve in a single setting. After 100 cycles it gave the same coverage as complete pooling, 94.6% against 95.0%, with an interval score of 10.3 against 18.7. With no cycles seen it was too wide, with an interval score of 31.5 against 18.6, and after 300 cycles it was over-confident, with a coverage of 52% and an interval score of 43.3. The reason is that one diagnostic nearly determines the state of health here, and the observation model treats successive residuals as independent. Where the question concerns cells and chemistries, the hierarchy answers it. Where it is prediction, the hierarchy should be compared with complete pooling, and with a control that has no diagnostics, before it is adopted, in the way done here.

## Reproducible Results

The notebook is in the [bayesian-modelling](https://github.com/sambaiga/bayesian-modelling) repository and runs from the first cell to the last in the `uv` environment of the repository, with PyMC 5.25, ArviZ 0.22 and lets-plot 4.8, and downloads the feature file when it first runs. The model, the sampling, the scores and the figures are functions of the `bayes` package, which has unit tests, and the cells of the notebook only call them. The sampling settings are in the first cell, and setting the environment variable `HIER_FAST=1` runs the whole notebook with a few draws in a minute or two, to check that it works before the full run. The fits are stored in `cache/`, so a repeat of the full run reads them back.